[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/03_%E9%A1%A7%E5%AE%A2%E3%82%BB%E3%82%B0%E3%83%A1%E3%83%B3%E3%83%86%E3%83%BC%E3%82%B7%E3%83%A7%E3%83%B3_RFM.ipynb)

# 発展マーケ-03. 顧客セグメンテーション（RFM × K-meansクラスタリング）

> 🟢 Colab（ブラウザ）で実行できます。**最初に「① セットアップ」セルを実行**してください。
> 📌 **発展編（統計検定2級の先：準1級〜実務/データサイエンス）**。scikit-learn・statsmodels を使います（Colabは導入済み。ローカルは `uv add scikit-learn statsmodels`）。

顧客を **RFM**（Recency 最近性・Frequency 頻度・Monetary 金額）で数値化し、**K-means** で似た顧客をグループ分けします（教師なし学習＝多変量解析）。

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルに無ければ公開リポジトリ(raw)から読み込む共通関数
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
def load(name):
    p = f'../data/{name}'
    return pd.read_csv(p) if os.path.exists(p) else pd.read_csv(RAW + name)
cust = load('btob_customers.csv')   # 顧客データ
cust.head()

## 1. RFM指標を作る

- **R** Recency：最後に買ってからの日数（小さいほど良い）
- **F** Frequency：購入回数
- **M** Monetary：累計購入額

In [ ]:
基準日 = pd.Timestamp('2026-01-01')                                  # 分析の基準日
cust['Recency'] = (基準日 - pd.to_datetime(cust['最終購入日'])).dt.days  # 最終購入からの日数
cust['Frequency'] = cust['購入回数']                                 # 購入回数
cust['Monetary'] = cust['累計売上']                                  # 累計額
cust[['顧客ID','Recency','Frequency','Monetary']].head()

## 2. 標準化（単位をそろえる）

R(日)・F(回)・M(円) は単位が違うので、**標準化（z得点）**してから距離を測ります。

In [ ]:
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(cust[['Recency','Frequency','Monetary']])  # 平均0・分散1に
print('標準化後の形:', X.shape)

## 3. K-means でクラスタリング

似たRFMの顧客を4グループに分けます（k=4）。

In [ ]:
from sklearn.cluster import KMeans
km = KMeans(n_clusters=4, random_state=0, n_init=10)   # 4グループに分ける
cust['cluster'] = km.fit_predict(X)                    # 各顧客にクラスタ番号を付与
print(cust['cluster'].value_counts().sort_index())     # クラスタごとの人数

## 4. クラスタの意味を読む（セグメント命名）

各クラスタの R/F/M の平均を見て、ビジネス上の意味をつけます。

In [ ]:
prof = cust.groupby('cluster')[['Recency','Frequency','Monetary']].mean().round(0)
print(prof)
print('\n読み方の例:')
print(' F・Mが高くRが小 → 優良顧客 / Rが大きい → 離反 / F・Mが小さくRも小 → 新規')

> 💡 クラスタ番号そのものに意味はありません。**R/F/Mの平均プロフィールで解釈**し、
「優良・一般・新規・離反」のような名前をつけて施策につなげます。

## 5. 2次元で可視化（PCA）

3次元(R,F,M)を主成分分析(PCA)で2次元に圧縮して散布図にします。

In [ ]:
from sklearn.decomposition import PCA
xy = PCA(n_components=2).fit_transform(X)              # 2次元へ圧縮
plt.scatter(xy[:,0], xy[:,1], c=cust['cluster'], cmap='tab10', alpha=0.6)  # 色=クラスタ
plt.title('顧客セグメント（PCAで2次元化）'); plt.xlabel('第1主成分'); plt.ylabel('第2主成分')
plt.show()

## 🧭 まとめ
- RFMで顧客を数値化 → 標準化 → K-meansで自動グループ分け。
- セグメントごとに施策を変える（優良=維持・特別対応、離反=掘り起こし、新規=育成）。

---
## 🏆 練習問題

**問1.** `k=3` で分け直し、R/F/M平均からセグメントを解釈しよう。

**問2.** エルボー法（k=1〜8 の `inertia_`）を折れ線にして、適切な k を考えよう。

**問3.** 各クラスタの `業界` 構成を `crosstab` で見よう。偏りはある？

**問4.** 最も Recency が大きいクラスタ（＝離反候補）の `顧客ID` を抽出しよう。

In [ ]:
# 問1〜4


> 🔑 **解答例は別ページ**（ネタバレ防止）👉 **[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/03_%E9%A1%A7%E5%AE%A2%E3%82%BB%E3%82%B0%E3%83%A1%E3%83%B3%E3%83%86%E3%83%BC%E3%82%B7%E3%83%A7%E3%83%B3_RFM.md)**